In [1]:
import pandas as pd

from credit_scoring import config as cfg

df = pd.read_parquet(cfg.REFERENCE_DIR / "train.parquet")
df.shape

(184506, 20)

In [2]:
df[cfg.TARGET].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [3]:
df.isna().mean().sort_values(ascending=False).loc[lambda s: s > 0].round(4)

EXT_SOURCE_1       0.5643
EXT_SOURCE_3       0.1978
EXT_SOURCE_2       0.0022
AMT_GOODS_PRICE    0.0008
AMT_ANNUITY        0.0000
dtype: float64

In [4]:
s = df["DAYS_EMPLOYED"]
print(s.describe())
print(s.value_counts().head(3))
print("positivos:", (s > 0).mean().round(4))

sentinela = s == 365243
print(df.groupby(sentinela)[cfg.TARGET].mean())
print(df.loc[sentinela, "NAME_INCOME_TYPE"].value_counts())

count    184506.000000
mean      63479.195939
std      140996.390082
min      -17583.000000
25%       -2762.750000
50%       -1215.000000
75%        -291.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64
DAYS_EMPLOYED
 365243    33056
-212         101
-207          99
Name: count, dtype: int64
positivos: 0.1792
DAYS_EMPLOYED
False    0.086841
True     0.052729
Name: TARGET, dtype: float64
NAME_INCOME_TYPE
Pensioner     33044
Unemployed       12
Name: count, dtype: int64


In [5]:
idade = -df["DAYS_BIRTH"] / 365.25
idade.describe()

count    184506.000000
mean         43.897632
std          11.938330
min          21.015743
25%          33.998631
50%          43.107461
75%          53.861054
max          68.993840
Name: DAYS_BIRTH, dtype: float64

In [6]:
renda = df["AMT_INCOME_TOTAL"]
print(renda.describe(percentiles=[0.5, 0.99, 0.999]))
cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "NAME_INCOME_TYPE", cfg.TARGET]
df.loc[renda.nlargest(5).index, cols]

count    1.845060e+05
mean     1.691125e+05
std      2.935670e+05
min      2.565000e+04
50%      1.485000e+05
99%      4.725000e+05
99.9%    9.000000e+05
max      1.170000e+08
Name: AMT_INCOME_TOTAL, dtype: float64


,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,TARGET
106375,117000000.0,562491.0,26194.5,Working,1
46331,18000090.0,675000.0,69295.5,Commercial associate,0
61421,9000000.0,1431531.0,132601.5,Working,0
56606,6750000.0,790830.0,52978.5,Working,0
50105,4500000.0,450000.0,47749.5,Working,0


In [7]:
for c in df.select_dtypes("object").columns:
    print(df[c].value_counts(dropna=False), "\n")

NAME_CONTRACT_TYPE
Cash loans         166916
Revolving loans     17590
Name: count, dtype: int64 

CODE_GENDER
F      121421
M       63084
XNA         1
Name: count, dtype: int64 

FLAG_OWN_CAR
N    121834
Y     62672
Name: count, dtype: int64 

FLAG_OWN_REALTY
Y    127917
N     56589
Name: count, dtype: int64 

NAME_INCOME_TYPE
Working                 95164
Commercial associate    43050
Pensioner               33052
State servant           13211
Unemployed                 12
Businessman                 9
Student                     6
Maternity leave             2
Name: count, dtype: int64 

NAME_EDUCATION_TYPE
Secondary / secondary special    131108
Higher education                  44747
Incomplete higher                  6257
Lower secondary                    2299
Academic degree                      95
Name: count, dtype: int64 

NAME_FAMILY_STATUS
Married                 117963
Single / not married     27244
Civil marriage           17850
Separated                11819
Widow     

C:\Users\bruno\AppData\Local\Temp\ipykernel_454152\2321927290.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes("object").columns:


In [8]:
df[cfg.ID_COL].duplicated().sum()

np.int64(0)

In [9]:
print("parcela >= crédito:", (df["AMT_ANNUITY"] >= df["AMT_CREDIT"]).sum())
print("bem > crédito:", (df["AMT_GOODS_PRICE"] > df["AMT_CREDIT"]).mean().round(4))

parcela >= crédito: 0
bem > crédito: 0.0009


In [10]:
# 1. Categóricas completas + numéricas não inspecionadas
pd.set_option("display.max_rows", 100)
for c in ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS"]:
    print(df[c].value_counts(dropna=False), "\n")
df[
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "CNT_CHILDREN", "REGION_RATING_CLIENT"]
].describe()

NAME_INCOME_TYPE
Working                 95164
Commercial associate    43050
Pensioner               33052
State servant           13211
Unemployed                 12
Businessman                 9
Student                     6
Maternity leave             2
Name: count, dtype: int64 

NAME_EDUCATION_TYPE
Secondary / secondary special    131108
Higher education                  44747
Incomplete higher                  6257
Lower secondary                    2299
Academic degree                      95
Name: count, dtype: int64 

NAME_FAMILY_STATUS
Married                 117963
Single / not married     27244
Civil marriage           17850
Separated                11819
Widow                     9629
Unknown                      1
Name: count, dtype: int64 



,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,CNT_CHILDREN,REGION_RATING_CLIENT
count,80388.000000,1.841080e+05,148006.000000,184506.000000,184506.000000
mean,0.502010,5.144528e-01,0.511244,0.417266,2.051597
std,0.211521,1.909477e-01,0.194725,0.719496,0.508675
min,0.014568,8.173617e-08,0.000527,0.000000,1.000000
25%,0.333748,3.924548e-01,0.370650,0.000000,2.000000
50%,0.506069,5.659319e-01,0.535276,0.000000,2.000000
75%,0.675217,6.635634e-01,0.669057,1.000000,2.000000
max,0.951624,8.549997e-01,0.896010,19.000000,3.000000


In [11]:
# 2. Coerência emprego x idade + linha do 117M
reais = df["DAYS_EMPLOYED"] != 365243
print("emprego > idade:", (df.loc[reais, "DAYS_EMPLOYED"] < df.loc[reais, "DAYS_BIRTH"]).sum())
df.loc[
    df["AMT_INCOME_TOTAL"].nlargest(3).index,
    ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "NAME_INCOME_TYPE", cfg.TARGET],
]

emprego > idade: 0


,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,NAME_INCOME_TYPE,TARGET
106375,117000000.0,562491.0,26194.5,Working,1
46331,18000090.0,675000.0,69295.5,Commercial associate,0
61421,9000000.0,1431531.0,132601.5,Working,0
